# SARIMAX

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. 

## 1. Symmetric protocol ($X = X'$)
Training and testing at 5-minute resolution with all channels. Writes `route1` (point + modular quantiles) and `route2` (integrated quantiles) CSVs for Dec 24 and Dec 30, 2024.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# SARIMAX symmetric evaluation at 5-minute multivariate resolution
# FEATURE-ALIGNED + STANDARDIZED ROUTE1 + RECURSIVE TEST FORECAST VERSION
#
# Train: V1 only
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2
#
# Route1:
#   - point SARIMAX trained ONCE on V1 only
#   - forward one-step prediction on validation
#   - NO retraining inside validation
#   - residual buckets built from validation residuals
#
# Route2:
#   - probabilistic SARIMAX via recursive one-step forecast intervals
#
# Forecasting mode alignment:
#   - test forecasting is recursive, one step at a time
# ============================


import os
import time
import warnings
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# =========================
# 0) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive"
os.makedirs(OUT_DIR, exist_ok=True)

MERGED_ROUTE1 = os.path.join(OUT_DIR, "merged_route1_point_uq_series_sarimax_symmetric_5min.csv")
MERGED_ROUTE2 = os.path.join(OUT_DIR, "merged_route2_predictions_prob_sarimax_symmetric_5min.csv")

# =========================
# 1) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 2) Forecast settings
# =========================
STEPS_PER_HOUR = 12
HORIZON_HOURS = 24
PRED_LEN = HORIZON_HOURS * STEPS_PER_HOUR   # 288

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
ALPHA_CI = 0.20

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

# =========================
# 3) Stable SARIMAX settings
# =========================
# simpler order because explicit lag features are already included
ORDER = (1, 0, 1)
SEASONAL_ORDER = (0, 0, 0, 0)

EXOG_COLS = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
    "lag_5min",
    "lag_10min",
    "lag_1h",
    "lag_24h",
    "ma_1h",
    "ma_3h",
]

VAL_STRIDE = 1   # keep 1 for dense standardized Route1 residuals

# =========================
# 4) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)

    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)

    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def fit_sarimax(y_train, X_train):
    print(f"  [SARIMAX] start fit | n={len(y_train)} | exog_dim={X_train.shape[1]}")
    model = SARIMAX(
        endog=y_train,
        exog=X_train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
        simple_differencing=True
    )

    t0 = time.time()
    res = model.fit(
        disp=False,
        method="lbfgs",
        maxiter=50
    )
    print(f"  [SARIMAX] fit done in {time.time() - t0:.2f}s")
    return res

def one_step_quantiles_from_state(res_state, X_next, alpha=0.20):
    f1 = res_state.get_forecast(steps=1, exog=X_next)
    mean = float(np.asarray(f1.predicted_mean).ravel()[0])

    ci = f1.conf_int(alpha=alpha)
    ci = np.asarray(ci, dtype=float).reshape(1, 2)
    lower = float(ci[0, 0])
    upper = float(ci[0, 1])

    z = float(norm.ppf(1 - alpha / 2))
    sigma = max((upper - lower) / (2.0 * z), 1e-9)

    q10 = mean + norm.ppf(0.10) * sigma
    q50 = mean
    q90 = mean + norm.ppf(0.90) * sigma

    aa, bb, cc = enforce_non_crossing(
        np.array([q10]), np.array([q50]), np.array([q90])
    )
    return float(aa[0]), float(bb[0]), float(cc[0])

def recursive_test_forecast_sarimax(res_trained, test_df, alpha=0.20):
    """
    Recursive forecasting mode alignment:
    - forecast one step at a time
    - update state with predicted q50
    - repeat for full horizon
    """
    res_state = res_trained

    q10_list, q50_list, q90_list = [], [], []

    for h in range(len(test_df)):
        X_next = test_df[EXOG_COLS].iloc[h:h+1].to_numpy(dtype=float)

        q10, q50, q90 = one_step_quantiles_from_state(res_state, X_next, alpha=alpha)

        q10_list.append(q10)
        q50_list.append(q50)
        q90_list.append(q90)

        # recursive update with predicted median
        res_state = res_state.append(
            endog=np.array([q50], dtype=float),
            exog=X_next,
            refit=False
        )

    return (
        np.asarray(q10_list, dtype=float),
        np.asarray(q50_list, dtype=float),
        np.asarray(q90_list, dtype=float),
    )

def forward_one_step_validation_predictions(res_trained, val_df, stride=1):
    """
    Standardized Route1:
    - start from the model trained on V1 only
    - forecast validation one step ahead sequentially
    - update state with ACTUAL validation observation
    - no refit inside validation
    """
    preds = []
    actuals = []
    ts_list = []

    res_state = res_trained

    for j in range(len(val_df)):
        if j % 1000 == 0:
            print(f"  validation rolling progress: {j}/{len(val_df)}")

        X_next = val_df[EXOG_COLS].iloc[j:j+1].to_numpy(dtype=float)

        try:
            f1 = res_state.get_forecast(steps=1, exog=X_next)
            mu = float(np.asarray(f1.predicted_mean).ravel()[0])
        except Exception as e:
            print(f"  validation forecast failed at j={j}: {repr(e)}")
            continue

        y_t = float(val_df["power_w"].iloc[j])
        ts = val_df["timestamp"].iloc[j]

        if (j % stride) == 0:
            preds.append(mu)
            actuals.append(y_t)
            ts_list.append(ts)

        try:
            res_state = res_state.append(
                endog=np.array([y_t], dtype=float),
                exog=X_next,
                refit=False
            )
        except Exception as e:
            print(f"  append failed at j={j}: {repr(e)}")
            break

    out = pd.DataFrame({
        "timestamp": ts_list,
        "actual": actuals,
        "pred": preds
    })
    return out

def build_route1_residual_buckets_from_val(res_trained, val_df):
    df_cal = forward_one_step_validation_predictions(res_trained, val_df, stride=VAL_STRIDE)

    if df_cal.empty:
        raise ValueError("Validation predictions are empty. Check SARIMAX stability.")

    df_cal["residual"] = df_cal["actual"] - df_cal["pred"]

    clip_val = np.percentile(np.abs(df_cal["residual"].to_numpy()), 99)
    df_cal["residual"] = np.clip(df_cal["residual"], -clip_val, clip_val)

    df_cal["slot_5min"] = df_cal["timestamp"].dt.hour * 12 + (df_cal["timestamp"].dt.minute // 5)
    df_cal["dayofweek"] = df_cal["timestamp"].dt.dayofweek

    res_q = (
        df_cal.groupby(["slot_5min", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    sigma_global = float(np.std(df_cal["residual"].to_numpy()))
    sigma_global = max(sigma_global, 1e-6)

    return res_q, sigma_global, df_cal

def apply_route1_buckets(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["slot_5min"] = tmp["timestamp"].dt.hour * 12 + (tmp["timestamp"].dt.minute // 5)
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["slot_5min", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# =========================
# 5) Load V1 and V2
# =========================
print("Loading V1 and V2 at 5-minute resolution...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min samples:", len(v1_5m))
print("✅ V2 5-min samples:", len(v2_5m))

# =========================
# 6) Symmetric split
# =========================
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_5m.copy().reset_index(drop=True)
val_df = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train samples:", len(train_df))
print("✅ Val samples  :", len(val_df))
print("✅ Test samples :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} points, expected {PRED_LEN}.")

# =========================
# Skip rerun if outputs exist
# =========================
if os.path.exists(MERGED_ROUTE1) and os.path.exists(MERGED_ROUTE2):
    print("\n✅ Final merged CSVs already exist. Skipping full pipeline.")
    print("Route1:", MERGED_ROUTE1)
    print("Route2:", MERGED_ROUTE2)

else:
    # =========================
    # 7) Train point SARIMAX on V1 only
    # =========================
    print("\nTraining SARIMAX on V1 only...")
    y_train_full = train_df["power_w"].to_numpy(dtype=float)
    X_train_full = train_df[EXOG_COLS].to_numpy(dtype=float)

    t0 = time.time()
    sarimax_point = fit_sarimax(y_train_full, X_train_full)
    print(f"Point model fit time: {time.time() - t0:.2f}s | train_samples={len(y_train_full)}")

    # =========================
    # 8) Build Route1 residual buckets
    # =========================
    print("\nBuilding Route1 residual buckets from validation using forward rolling one-step prediction...")
    res_q, sigma_global, df_res = build_route1_residual_buckets_from_val(sarimax_point, val_df)

    resq_path = os.path.join(OUT_DIR, "sarimax_route1_residual_quantiles_from_v2val_5min.csv")
    resid_path = os.path.join(OUT_DIR, "sarimax_route1_residuals_from_v2val_5min.csv")

    res_q.to_csv(resq_path, index=False)
    df_res.to_csv(resid_path, index=False)

    print("✅ Saved residual buckets:", resq_path)
    print("✅ Saved residual table  :", resid_path)
    print(f"Fallback sigma_global: {sigma_global:.4f}")

    # =========================
    # 9) Test on Dec 24 and Dec 30
    # =========================
    print("\nTesting on Dec 24 and Dec 30 with recursive one-step forecasting...")

    r1_uq_day_files = []
    r2_prob_day_files = []

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print("\n" + "=" * 70)
        print(f"Test day {k}: {day_start.date()}")

        day_end = day_start + pd.Timedelta(days=1)

        test_df = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        y_true = test_df["power_w"].to_numpy(dtype=float)

        # Recursive Route2 and recursive point forecast
        q10_prob, q50_prob, q90_prob = recursive_test_forecast_sarimax(
            sarimax_point, test_df, alpha=ALPHA_CI
        )
        point_pred = q50_prob.copy()

        # Route1: apply standardized residual buckets to recursive point prediction
        p10_uq, p50_uq, p90_uq = apply_route1_buckets(
            test_df["timestamp"].values, point_pred, res_q
        )

        out_r1 = pd.DataFrame({
            "timestamp": test_df["timestamp"].values,
            "actual": y_true,
            "point_pred": point_pred,
            "q10": p10_uq,
            "q50": p50_uq,
            "q90": p90_uq
        })
        p_r1 = os.path.join(OUT_DIR, f"sarimax_route1_uq_symmetric_5min_day{k}.csv")
        out_r1.to_csv(p_r1, index=False)
        r1_uq_day_files.append(p_r1)

        out_r2 = pd.DataFrame({
            "timestamp": test_df["timestamp"].values,
            "actual": y_true,
            "q10": q10_prob,
            "q50": q50_prob,
            "q90": q90_prob
        })
        p_r2 = os.path.join(OUT_DIR, f"sarimax_route2_prob_symmetric_5min_day{k}.csv")
        out_r2.to_csv(p_r2, index=False)
        r2_prob_day_files.append(p_r2)

        print("✅ Saved:")
        print("  Route1_UQ   :", p_r1)
        print("  Route2_Prob :", p_r2)

    # =========================
    # 10) Merge outputs
    # =========================
    print("\nMerging outputs...")

    merged_route1_df = merge_daily_route1(r1_uq_day_files, MERGED_ROUTE1)
    merged_route2_df = merge_daily_route2(r2_prob_day_files, MERGED_ROUTE2)

    print("✅ Merged files written:")
    print("  Route1:", MERGED_ROUTE1)
    print("  Route2:", MERGED_ROUTE2)
    print("  OUT_DIR:", OUT_DIR)

    print("\nRoute1 head:")
    print(merged_route1_df.head())

    print("\nRoute2 head:")
    print(merged_route2_df.head())

## 2. Asymmetric protocol ($X \neq X'$)
Training on the full multivariate history; deployment on hourly data with temperature and calendar features only. Voltage, current and PV generation are filled with hour-of-day means from $D_1$.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# SARIMAX ASYMMETRIC EVALUATION
# NATIVE HOURLY DEPLOYMENT VERSION
#
# Unified asymmetric protocol:
#   - Train source: V1 only
#   - Validation source: V2 excluding Dec 24 and Dec 30
#   - Test source: Dec 24 and Dec 30 from V2
#   - Evaluation resolution: HOURLY
#   - Deployment-visible covariates: temperature + time only
#   - Missing variables are filled using V1-derived priors
#
# Route1:
#   point SARIMAX + post-hoc residual buckets from degraded hourly V2 validation
#
# Route2:
#   integrated probabilistic SARIMAX intervals directly from hourly forecast intervals
#
# This version is HOURLY all the way for SARIMAX evaluation.
# It removes the old inconsistency where hourly deployment inputs
# were expanded back to 5-minute rows before prediction.
# ============================


import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# =========================
# 0) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly"
os.makedirs(OUT_DIR, exist_ok=True)

MERGED_ROUTE1 = os.path.join(OUT_DIR, "merged_route1_point_uq_series_sarimax_asymmetric_native_hourly.csv")
MERGED_ROUTE2 = os.path.join(OUT_DIR, "merged_route2_predictions_prob_sarimax_asymmetric_native_hourly.csv")
META_PATH = os.path.join(OUT_DIR, "run_metadata.json")

# =========================
# 1) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 2) Forecast settings
# =========================
Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
ALPHA80 = 0.20

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

EXPECTED_HOURLY_POINTS_PER_DAY = 24

# =========================
# 3) SARIMAX settings
# =========================
# Hourly daily seasonality = 24
ORDER = (2, 0, 2)
SEASONAL_ORDER = (1, 0, 0, 24)

EXOG_COLS = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "lag_1h",
    "lag_2h",
    "lag_24h",
    "ma_3h",
    "ma_24h",
]

MAX_OPT_ITERS = 80
REFIT_EVERY_HOURS = 24
VAL_MIN_HISTORY_HOURS = 24 * 7
VAL_STRIDE_HOURS = 1
MAX_CAL_WINDOWS = 1200

# =========================
# 4) Metadata
# =========================
meta = {
    "train_source": "V1 only",
    "validation_source": "V2 excluding Dec 24 and Dec 30",
    "test_days": [str(x.date()) for x in TEST_DAYS],
    "raw_input_resolution": "5min",
    "training_resolution": "hourly",
    "deployment_resolution": "hourly",
    "target": "power_w",
    "exog_cols": EXOG_COLS,
    "sarimax_order": ORDER,
    "sarimax_seasonal_order": SEASONAL_ORDER,
    "mode": "native_hourly_degraded_deployment",
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved metadata:", META_PATH)

# =========================
# 5) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def cast_numeric_float32(df):
    df = df.copy()
    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[num_cols] = df[num_cols].astype("float32")
    return df

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)

    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)

    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def maybe_limit_indices(indices, max_windows):
    if len(indices) <= max_windows:
        return indices
    pick = np.linspace(0, len(indices) - 1, max_windows).round().astype(int)
    pick = np.unique(pick)
    return [indices[i] for i in pick]

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    return cast_numeric_float32(out)

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    return cast_numeric_float32(df)

def aggregate_to_hourly(df_5m):
    df = df_5m.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ts_hour"] = df["timestamp"].dt.floor("H")

    ag = (
        df.groupby("ts_hour", as_index=False)
          .agg({
              "power_w": "mean",
              "temp_c": "mean",
              "voltage_v": "mean",
              "amperage_a": "mean",
              "generation_w": "mean",
          })
          .rename(columns={"ts_hour": "timestamp"})
    )

    ag["hour"] = ag["timestamp"].dt.hour
    ag["dayofweek"] = ag["timestamp"].dt.dayofweek
    return cast_numeric_float32(ag)

def add_engineered_features_hourly(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek

    df["lag_1h"] = df["power_w"].shift(1)
    df["lag_2h"] = df["power_w"].shift(2)
    df["lag_24h"] = df["power_w"].shift(24)

    df["ma_3h"] = df["power_w"].shift(1).rolling(window=3, min_periods=1).mean()
    df["ma_24h"] = df["power_w"].shift(1).rolling(window=24, min_periods=1).mean()

    df = df.bfill().ffill()
    return cast_numeric_float32(df)

def build_hourly_priors_from_train(train_h):
    priors_hour = (
        train_h.groupby("hour")[["voltage_v", "amperage_a", "generation_w"]]
               .median()
               .reset_index()
    )

    priors_global = {
        "temp_c_fallback": float(train_h["temp_c"].median()),
        "voltage_v": float(train_h["voltage_v"].median()),
        "amperage_a": float(train_h["amperage_a"].median()),
        "generation_w": float(train_h["generation_w"].median()),
        "lag_1h": float(train_h["power_w"].shift(1).median()),
        "lag_2h": float(train_h["power_w"].shift(2).median()),
        "lag_24h": float(train_h["power_w"].shift(24).median()),
        "ma_3h": float(train_h["power_w"].shift(1).rolling(3, min_periods=1).mean().median()),
        "ma_24h": float(train_h["power_w"].shift(1).rolling(24, min_periods=1).mean().median()),
    }

    return cast_numeric_float32(priors_hour), priors_global

def build_deployment_hourly_frame(hourly_df, priors_hour_df, priors_global):
    """
    Deployment-visible:
      - temp_c
      - hour
      - dayofweek

    Deployment-unavailable variables:
      - voltage_v, amperage_a, generation_w -> V1 hourly priors
      - lag_1h, lag_2h, lag_24h, ma_3h, ma_24h -> V1 global priors
    """
    df = hourly_df.copy().reset_index(drop=True)
    pri_map = priors_hour_df.set_index("hour")[["voltage_v", "amperage_a", "generation_w"]].to_dict("index")

    rows = []
    for _, r in df.iterrows():
        ts = pd.Timestamp(r["timestamp"])
        h = int(ts.hour)
        dow = int(ts.dayofweek)

        pri = pri_map.get(h, {})
        rows.append({
            "timestamp": ts,
            "power_w": float(r["power_w"]) if "power_w" in df.columns else np.nan,
            "temp_c": float(r["temp_c"]) if pd.notna(r["temp_c"]) else float(priors_global["temp_c_fallback"]),
            "voltage_v": float(pri.get("voltage_v", priors_global["voltage_v"])),
            "amperage_a": float(pri.get("amperage_a", priors_global["amperage_a"])),
            "generation_w": float(pri.get("generation_w", priors_global["generation_w"])),
            "hour": h,
            "dayofweek": dow,
            "lag_1h": float(priors_global["lag_1h"]),
            "lag_2h": float(priors_global["lag_2h"]),
            "lag_24h": float(priors_global["lag_24h"]),
            "ma_3h": float(priors_global["ma_3h"]),
            "ma_24h": float(priors_global["ma_24h"]),
        })

    out = pd.DataFrame(rows)
    return cast_numeric_float32(out)

def fit_sarimax(y_train, X_train):
    print(f"  [SARIMAX] start fit | n={len(y_train)} | exog_dim={X_train.shape[1]}")
    model = SARIMAX(
        endog=y_train,
        exog=X_train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
        simple_differencing=True
    )

    iter_counter = {"i": 0}

    def _callback(params):
        iter_counter["i"] += 1
        if iter_counter["i"] % 5 == 0 or iter_counter["i"] == 1:
            print(f"    optimizer iteration {iter_counter['i']}/{MAX_OPT_ITERS}")

    t0 = time.time()
    res = model.fit(
        disp=False,
        method="lbfgs",
        maxiter=MAX_OPT_ITERS,
        callback=_callback
    )
    print(f"  [SARIMAX] fit done in {time.time() - t0:.2f}s")
    return res

def sarimax_forecast_quantiles_hourly(res, X_future_h, alpha=0.20):
    f = res.get_forecast(steps=len(X_future_h), exog=X_future_h)
    mean_h = np.asarray(f.predicted_mean, dtype=float).ravel()

    ci = f.conf_int(alpha=alpha)
    ci = np.asarray(ci, dtype=float)
    lower_h = ci[:, 0]
    upper_h = ci[:, 1]

    z = float(norm.ppf(1 - alpha / 2))
    sigma_h = (upper_h - lower_h) / (2.0 * z)
    sigma_h = np.clip(sigma_h, 1e-9, None)

    q10_h = mean_h + norm.ppf(0.10) * sigma_h
    q50_h = mean_h
    q90_h = mean_h + norm.ppf(0.90) * sigma_h

    return enforce_non_crossing(q10_h, q50_h, q90_h)

def apply_route1_buckets_hourly(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["hour"] = tmp["timestamp"].dt.hour
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["hour", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

def rolling_one_step_hourly_predictions(
    val_h,
    train_h,
    priors_hour,
    priors_global,
    min_history=VAL_MIN_HISTORY_HOURS,
    stride_hours=VAL_STRIDE_HOURS,
    refit_every_hours=REFIT_EVERY_HOURS,
):
    """
    For each validation hour j:
      - fit/refit on all hourly history available before that hour
      - build degraded deployment exog for that one hour
      - forecast next hourly point
    """
    preds = []
    actuals = []
    ts_list = []

    all_hour_idx = list(range(min_history, len(val_h), stride_hours))
    all_hour_idx = maybe_limit_indices(all_hour_idx, MAX_CAL_WINDOWS)

    print(f"Validation hourly windows used: {len(all_hour_idx)}")

    res = None
    for n_idx, j in enumerate(all_hour_idx, start=1):
        hist_part = val_h.iloc[:j].copy()
        full_hist = pd.concat([train_h, hist_part], ignore_index=True).reset_index(drop=True)

        if (res is None) or ((n_idx - 1) % refit_every_hours == 0):
            y_hist = full_hist["power_w"].to_numpy(dtype=float)
            X_hist = full_hist[EXOG_COLS].to_numpy(dtype=float)
            print(f"  Refit {n_idx}/{len(all_hour_idx)} | hist_rows={len(full_hist)}")
            res = fit_sarimax(y_hist, X_hist)

        row_true = val_h.iloc[[j]].copy()
        dep_row = build_deployment_hourly_frame(row_true, priors_hour, priors_global)
        X_future = dep_row[EXOG_COLS].to_numpy(dtype=float)

        fc = res.get_forecast(steps=1, exog=X_future)
        pred = float(np.asarray(fc.predicted_mean, dtype=float).ravel()[0])

        preds.append(pred)
        actuals.append(float(row_true["power_w"].iloc[0]))
        ts_list.append(pd.Timestamp(row_true["timestamp"].iloc[0]))

        if n_idx % 50 == 0 or n_idx == len(all_hour_idx):
            print(f"  Calibration progress: {n_idx}/{len(all_hour_idx)}")

    out = pd.DataFrame({
        "timestamp": ts_list,
        "actual": actuals,
        "point_pred": preds,
    })
    return out

def build_route1_residual_buckets_from_val_asymmetric_hourly(
    val_h,
    train_h,
    priors_hour,
    priors_global,
):
    df_pred = rolling_one_step_hourly_predictions(
        val_h=val_h,
        train_h=train_h,
        priors_hour=priors_hour,
        priors_global=priors_global,
    )

    df_pred["hour"] = df_pred["timestamp"].dt.hour
    df_pred["dayofweek"] = df_pred["timestamp"].dt.dayofweek
    df_pred["residual"] = df_pred["actual"] - df_pred["point_pred"]

    sigma_global = float(df_pred["residual"].std(ddof=1)) if len(df_pred) > 1 else 0.0

    res_q = (
        df_pred.groupby(["hour", "dayofweek"])["residual"]
               .quantile([Q_LO, Q_MID, Q_HI])
               .unstack()
               .reset_index()
    )
    res_q.columns = ["hour", "dayofweek", "rq10", "rq50", "rq90"]

    return res_q, sigma_global, df_pred

# =========================
# 6) Load raw data
# =========================
print("Loading raw V1 and V2 at 5-minute resolution...")

v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

v1_h = add_engineered_features_hourly(aggregate_to_hourly(v1_5m))
v2_h = add_engineered_features_hourly(aggregate_to_hourly(v2_5m))

full_h = pd.concat([v1_h, v2_h], ignore_index=True)
full_h = full_h.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

print("V1 hourly:", len(v1_h))
print("V2 hourly:", len(v2_h))
print("Full hourly:", len(full_h))

# =========================
# 7) Split
# =========================
test_mask = v2_h["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_h.copy().reset_index(drop=True)
val_df = v2_h[~test_mask].copy().reset_index(drop=True)
test_df_hourly_all = v2_h[test_mask].copy().reset_index(drop=True)

print("\nSplit sizes:")
print("Train rows:", len(train_df))
print("Val rows  :", len(val_df))
print("Test rows :", len(test_df_hourly_all))

for d in TEST_DAYS:
    cur = test_df_hourly_all[test_df_hourly_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {d.date()} has {len(cur)} hourly rows, expected 24.")

# =========================
# 8) Build V1 priors
# =========================
print("\nBuilding V1 priors for deployment-time missing variables...")
priors_hour, priors_global = build_hourly_priors_from_train(train_df)

pd.DataFrame(priors_hour).to_csv(
    os.path.join(OUT_DIR, "v1_hourly_priors_for_missing_vars.csv"),
    index=False
)
pd.DataFrame([priors_global]).to_csv(
    os.path.join(OUT_DIR, "v1_global_priors_for_missing_vars.csv"),
    index=False
)

# =========================
# 9) Skip rerun if outputs exist
# =========================
if os.path.exists(MERGED_ROUTE1) and os.path.exists(MERGED_ROUTE2):
    print("\n✅ Final merged CSVs already exist. Skipping full pipeline.")
    print("Route1:", MERGED_ROUTE1)
    print("Route2:", MERGED_ROUTE2)

else:
    # =========================
    # 10) Train point SARIMAX on V1 hourly rich regime
    # =========================
    print("\nTraining SARIMAX on V1 only, hourly rich regime...")

    y_train_full = train_df["power_w"].to_numpy(dtype=float)
    X_train_full = train_df[EXOG_COLS].to_numpy(dtype=float)

    t0 = time.time()
    sarimax_point = fit_sarimax(y_train_full, X_train_full)
    print(f"Point model fit time: {time.time()-t0:.2f}s | train_samples={len(y_train_full)}")

    # =========================
    # 11) Route1 residual buckets from degraded hourly validation
    # =========================
    print("\nBuilding Route1 residual buckets from V2 validation in degraded hourly mode...")
    res_q, sigma_global, df_res = build_route1_residual_buckets_from_val_asymmetric_hourly(
        val_h=val_df,
        train_h=train_df,
        priors_hour=priors_hour,
        priors_global=priors_global
    )

    res_q.to_csv(os.path.join(OUT_DIR, "sarimax_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv"), index=False)
    df_res.to_csv(os.path.join(OUT_DIR, "sarimax_route1_residuals_from_v2val_asymmetric_hourly.csv"), index=False)

    print("✅ Saved Route1 residual buckets")
    print(f"Fallback sigma_global: {sigma_global:.4f}")

    # =========================
    # 12) Test on Dec 24 and Dec 30 in degraded hourly mode
    # =========================
    print("\nTesting on Dec 24 and Dec 30 in degraded hourly mode...")

    r1_uq_day_files = []
    r2_prob_day_files = []

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print("\n" + "=" * 80)
        print(f"Test day {k}: {day_start.date()}")

        day_end = day_start + pd.Timedelta(days=1)

        test_h = test_df_hourly_all[
            (test_df_hourly_all["timestamp"] >= day_start) &
            (test_df_hourly_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        if len(test_h) != EXPECTED_HOURLY_POINTS_PER_DAY:
            raise ValueError(f"Test day {day_start.date()} does not have 24 hourly rows.")

        dep_h = build_deployment_hourly_frame(test_h, priors_hour, priors_global)
        X_future_h = dep_h[EXOG_COLS].to_numpy(dtype=float)
        y_true = test_h["power_w"].to_numpy(dtype=float)

        # Route2: native hourly probabilistic SARIMAX
        q10_prob, q50_prob, q90_prob = sarimax_forecast_quantiles_hourly(
            sarimax_point,
            X_future_h,
            alpha=ALPHA80
        )
        point_pred = q50_prob.copy()

        out_r2 = pd.DataFrame({
            "timestamp": test_h["timestamp"].values,
            "actual": y_true,
            "q10": q10_prob,
            "q50": q50_prob,
            "q90": q90_prob,
        })
        p_r2 = os.path.join(OUT_DIR, f"sarimax_route2_prob_asymmetric_hourly_day{k}.csv")
        out_r2.to_csv(p_r2, index=False)
        r2_prob_day_files.append(p_r2)

        # Route1: post-hoc residual buckets
        p10_uq, p50_uq, p90_uq = apply_route1_buckets_hourly(
            test_h["timestamp"].values,
            point_pred,
            res_q
        )

        out_r1 = pd.DataFrame({
            "timestamp": test_h["timestamp"].values,
            "actual": y_true,
            "point_pred": point_pred,
            "q10": p10_uq,
            "q50": p50_uq,
            "q90": p90_uq,
        })
        p_r1 = os.path.join(OUT_DIR, f"sarimax_route1_uq_asymmetric_hourly_day{k}.csv")
        out_r1.to_csv(p_r1, index=False)
        r1_uq_day_files.append(p_r1)

        print(f"Saved Route1 day file: {p_r1}")
        print(f"Saved Route2 day file: {p_r2}")

    # =========================
    # 13) Merge final outputs
    # =========================
    print("\nMerging daily outputs...")
    merged_r1 = merge_daily_route1(r1_uq_day_files, MERGED_ROUTE1)
    merged_r2 = merge_daily_route2(r2_prob_day_files, MERGED_ROUTE2)

    print("✅ Route1 merged:", MERGED_ROUTE1, "| rows:", len(merged_r1))
    print("✅ Route2 merged:", MERGED_ROUTE2, "| rows:", len(merged_r2))

print("\nDone.")
print("Route1 merged output:", MERGED_ROUTE1)
print("Route2 merged output:", MERGED_ROUTE2)